In [1]:
import os

In [2]:
%pwd

'd:\\codes\\mlflow\\data-science-project\\reserch'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\codes\\mlflow\\data-science-project'

In [5]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class DataIngestionconfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [12]:
from src.datascienceproject.constants import *
from src.datascienceproject.utils.common import read_yaml, create_directories

In [21]:
class ConfiguationManager:
    def __init__(self, 
                 config_pathfile=CONFIG_FILE_PATH,
                 params_pathfile=PARAMS_FILE_PATH,
                 schema_pathfile=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_pathfile)
        self.params=read_yaml(params_pathfile)
        self.schema=read_yaml(schema_pathfile)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)->DataIngestionconfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config=DataIngestionconfig(root_dir=config.root_dir,
                                                  source_URL=config.source_URL,
                                                  local_data_file=config.local_data_file,
                                                  unzip_dir=config.unzip_dir)
        return data_ingestion_config

In [24]:
import os
import urllib.request as request
from src.datascienceproject import logger
import zipfile
class DataIngestion:
    def __init__(self,config:DataIngestionconfig):
        self.config=config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info("file alredy exists")

    def extract_zip_file(self):
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [25]:
try:
    config=ConfiguationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-09-12 17:56:56,584: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-12 17:56:56,587: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-12 17:56:56,590: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-12 17:56:56,592: INFO: common: created directory at: artifacts]
[2026-09-12 17:56:56,595: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-12 17:56:59,451: INFO: 591481306: artifacts/data_ingestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 16EC:3BEFEF:328161:52A3D5:6AA56132
x-github-edge-region: uksouth
Acc